# **Feature Engineering — H&M**

Construir as variáveis necessárias para analisar recorrência, características dos clientes, produtos e canais comerciais.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.options.display.max_columns = None
pd.options.display.float_format = "{:.3f}".format

In [3]:
DATA_DIR = Path("../data").resolve()

articles = pd.read_parquet(DATA_DIR / "articles_clean.parquet")
customers = pd.read_parquet(DATA_DIR / "customers_clean.parquet")
transactions = pd.read_parquet(DATA_DIR / "transactions_clean.parquet")

In [4]:
dataframes = {
    "articles": articles,
    "customers": customers,
    "transactions": transactions,
}

for name, dataframe in dataframes.items():
    print(f"{name}: {dataframe.shape}")
    display(dataframe.head())

articles: (105542, 24)


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,perceived_colour_value_id,perceived_colour_value_name,perceived_colour_master_id,perceived_colour_master_name,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,4,Dark,5,Black,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,3,Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,1,Dusty Light,9,White,1676,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,4,Dark,5,Black,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear"
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,3,Light,9,White,1339,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear"


customers: (1371980, 6)


,customer_id,fn,active,club_member_status,fashion_news_frequency,age
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.000,1.000,ACTIVE,REGULARLY,52


transactions: (31788324, 5)


,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.051,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.017,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.017,2


## Customer_daily
- Agregação de transações por cliente e data

In [5]:
## criação da tabela intermediária agregada de cliente + transação
customer_daily = transactions.groupby(["customer_id", "t_dat"], observed=True).agg(
    items_day = ("article_id", "size"),
    unique_articles_id = ("article_id", "nunique"),
    value_day = ("price", "sum")
).reset_index()

In [6]:

display(customer_daily.head())

## validação
print(transactions['customer_id'].nunique() == customer_daily['customer_id'].nunique())
print(len(transactions) == customer_daily["items_day"].sum())

,customer_id,t_dat,items_day,unique_articles_id,value_day
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,2018-12-27,3,3,0.110
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,2019-05-02,1,1,0.010
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,2019-05-25,2,1,0.102
3,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,2019-07-25,2,2,0.034
4,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,2019-09-18,1,1,0.044


True
True


## Customer_purchase
-- Agregação das compras de cada cliente por data

In [7]:
# Customer_purchase

customer_purchase = customer_daily.groupby("customer_id", observed=True).agg(
    first_purchase_date = ("t_dat", "min"),
    last_purchase_date = ("t_dat", "max"),
    purchase_days = ("t_dat", "size"),
    items_purchased = ("items_day", "sum"),
    value = ("value_day", "sum")
).reset_index()


In [8]:
## criar a coluna de segunda compra pra medir a recorrência
second_purchase = customer_daily[customer_daily.duplicated("customer_id")].drop_duplicates("customer_id").set_index("customer_id")["t_dat"]

customer_purchase ["second_purchase_date"] = customer_purchase["customer_id"].map(second_purchase)

## ordenar a coluna de data para melhorar visualização
ordem = ['customer_id', 'first_purchase_date', 'second_purchase_date', 'last_purchase_date',
       'purchase_days', 'items_purchased', 'value']
customer_purchase = customer_purchase[ordem]

In [9]:
## validação
print(len(customer_purchase) == transactions['customer_id'].nunique())
print(customer_purchase["customer_id"].duplicated().sum() == 0)
print(customer_purchase["items_purchased"].sum() == len(transactions))

True
True
True


In [10]:
## datas de final de observação e recência
observation_end = transactions["t_dat"].max()
reference_date = observation_end + pd.Timedelta(days=1)

print(f"Fim da observação: {observation_end}")
print(f"Data de referência: {reference_date}")

Fim da observação: 2020-09-22 00:00:00
Data de referência: 2020-09-23 00:00:00


In [11]:
## Criação de features temporais

customer_purchase["days_to_second_purchase"] = (customer_purchase["second_purchase_date"]- customer_purchase["first_purchase_date"]).dt.days.astype("Int64")

customer_purchase["recency_days"] = (reference_date - customer_purchase["last_purchase_date"]).dt.days.astype("Int64")

customer_purchase["available_followup_days"] = (observation_end - customer_purchase["first_purchase_date"]).dt.days.astype("Int64")

customer_purchase["is_recurring_observed"] = (customer_purchase["purchase_days"] >= 2)

customer_purchase["cohort_month"] = (customer_purchase["first_purchase_date"].dt.to_period("M").dt.to_timestamp())

## Customer_behavior

- transações agregadas por cliente pra medir o comportamento de compra e a utilização de cada canal


In [12]:
transactions.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.051,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.017,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.017,2


In [13]:
customer_behavior = transactions.groupby("customer_id").agg(
    unique_article = ("article_id", "nunique"),
    channels_used = ("sales_channel_id", "nunique"),
    channel_reference = ("sales_channel_id", "min")
)

In [14]:
## criação da do perfil de canal
customer_behavior["channel_profile"] = np.select(
    [
        customer_behavior["channels_used"].eq(2),
        customer_behavior["channel_reference"].eq(1),
        customer_behavior["channel_reference"].eq(2),
    ],
    [
        "MULTICHANNEL",
        "CHANNEL_1_ONLY",
        "CHANNEL_2_ONLY",
    ],
    default="UNKNOWN",
)

customer_behavior = customer_behavior.drop(columns="channel_reference")

In [15]:
## merge com a base de compras

customer_purchase = customer_purchase.merge(
    customer_behavior,
    on="customer_id",
    how="left",
    validate="one_to_one",
)

In [16]:
## validação
print(customer_purchase["customer_id"].duplicated().sum())
print(customer_purchase ["unique_article"].isnull().sum())
print(customer_purchase ["channels_used"].isnull().sum())
print(customer_purchase ["channel_profile"].isnull().sum())
print(customer_purchase["channels_used"].unique())


0
0
0
0
[2 1]


## customer_attributes 

- Atributos de cadastro a clientes, como faixa etária, pra permitir medidas de consumo

In [17]:
customer_attributes = customers.copy()

In [18]:
customer_attributes["age_group"] = pd.cut(
    customer_attributes["age"],
    bins=[15,25,35,45,55,65,np.inf],
    labels=["15-24","25-34","35-44","45-54","55-64","65+"],
    right=False
)




In [19]:
## validação do grupo e verificação de nulos ou ausentes antes do merge
print(customer_attributes["age_group"].isnull().sum())

15861


In [20]:
customer_attributes["age_group"] = (
    customer_attributes["age_group"]
    .cat.add_categories(["NOT_INFORMED"])
    .fillna("NOT_INFORMED")
)

In [21]:
## merge com as compras

customer_purchase = customer_purchase.merge(
    customer_attributes,
    on="customer_id",
    how="left",
    validate="one_to_one",
)

In [22]:
## validação
print(customer_purchase["customer_id"].duplicated().sum())
print(customer_attributes["age_group"].isnull().sum())
print(customer_attributes["age"].isnull().sum() == (customer_attributes["age_group"] == "NOT_INFORMED").sum())

0
0
True


## Customer_monthly

-- agregação de cliente em cada mês no qual houve compra pra avaliar a evolução temporal


In [23]:
## criação das colunas de agregação

customer_daily["purchase_month"] = (customer_daily["t_dat"].dt.to_period("M").dt.to_timestamp())

customer_monthly = (customer_daily.groupby(["customer_id", "purchase_month"],observed=True,sort=False, ).agg(
        purchase_days_month=("t_dat", "size"),
        items_purchased_month=("items_day", "sum"),
        observed_value_month=("value_day", "sum"),
    ).reset_index()
)

In [24]:
## merge com customer para coorte

customer_monthly = customer_monthly.merge(
    customer_purchase[["customer_id", "cohort_month"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)

In [25]:
customer_monthly["cohort_index"] = ((customer_monthly["purchase_month"].dt.year - customer_monthly["cohort_month"].dt.year) * 12 + ( customer_monthly["purchase_month"].dt.month - customer_monthly["cohort_month"].dt.month )).astype("Int64")

## Purchase_Stage

- Medidas para analisar a recorrência de compra dentre os clientes

In [26]:
transactions["purchase_month"] = (transactions["t_dat"].dt.to_period("M").dt.to_timestamp())

In [27]:
first_purchase_by_customer = (
    customer_purchase.set_index("customer_id")["first_purchase_date"]
)

first_purchase_mask = transactions["t_dat"].eq(
    transactions["customer_id"].map(first_purchase_by_customer)
)

In [28]:
transactions["purchase_stage"] = np.where(
    first_purchase_mask,
    "FIRST_OBSERVED_DATE",
    "SUBSEQUENT_DATE",
)

transactions["purchase_stage"] = (
    transactions["purchase_stage"].astype("category")
)

In [29]:
del first_purchase_by_customer, first_purchase_mask

In [ ]:
print(transactions["purchase_month"].isnull().sum())
print(transactions["purchase_stage"].isnull().sum())

0
0
False


In [ ]:
output_files = {
    "customer_purchase": DATA_DIR / "customer_purchase.parquet",
    "customer_monthly": DATA_DIR / "customer_monthly.parquet",
    "transactions_features": DATA_DIR / "transactions_features.parquet",
}

customer_purchase.to_parquet(
    output_files["customer_purchase"],
    index=False,
    compression="snappy",
)

customer_monthly.to_parquet(
    output_files["customer_monthly"],
    index=False,
    compression="snappy",
)

transactions.to_parquet(
    output_files["transactions_features"],
    index=False,
    compression="snappy",
)